# Notebook 52 — RAG Pipeline

Demonstrates `multigen.rag`: chunking strategies, random embedder, vector / BM25 /
hybrid indexes, citation tracking, retrieval feedback, and the end-to-end
`RAGPipeline`.  No external APIs are required.

In [ ]:
import sys, importlib.util
sys.path.insert(0, '../sdk')

def load(name):
    spec = importlib.util.spec_from_file_location(
        f'multigen.{name}', f'../sdk/multigen/{name}.py')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[f'multigen.{name}'] = mod
    spec.loader.exec_module(mod)
    return mod

rag = load('rag')
print('rag loaded OK')

## 1. FixedChunker vs SentenceChunker vs RecursiveChunker — chunk count comparison

In [ ]:
TEXT = (
    "The company reported record revenue of $12.4 billion in Q3. "
    "This was driven by strong growth in the cloud division. "
    "Operating margins improved to 28% year-over-year. "
    "Management expects continued momentum into Q4. "
    "Capital expenditures remained flat at $1.2 billion. "
    "The board approved a $500 million share buyback programme. "
    "Employee headcount grew by 3% globally. "
    "International revenue now accounts for 45% of total sales."
)

fixed     = rag.FixedChunker(chunk_size=120, overlap=20)
sentence  = rag.SentenceChunker(max_sentences=2, overlap_sentences=1)
recursive = rag.RecursiveChunker(chunk_size=120, overlap=20)

fc = fixed.chunk(TEXT, source='report')
sc = sentence.chunk(TEXT, source='report')
rc = recursive.chunk(TEXT, source='report')

print(f'FixedChunker     → {len(fc)} chunks')
print(f'SentenceChunker  → {len(sc)} chunks')
print(f'RecursiveChunker → {len(rc)} chunks')

print('\nFirst fixed chunk :', fc[0].text[:80])
print('First sentence chunk:', sc[0].text[:80])

## 2. RandomEmbedder — cosine similarity between similar vs dissimilar sentences

In [ ]:
import math

def cosine(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a)) or 1e-9
    nb = math.sqrt(sum(x*x for x in b)) or 1e-9
    return dot / (na * nb)

embedder = rag.RandomEmbedder(dim=64)

s1 = 'Revenue grew 20% year-over-year.'
s2 = 'Revenue increased 20% compared to last year.'   # semantically similar
s3 = 'The weather in London is often cloudy.'          # dissimilar

e1, e2, e3 = embedder.embed_batch([s1, s2, s3])

sim_similar    = cosine(e1, e2)
sim_dissimilar = cosine(e1, e3)

print(f'Cosine(s1, s2) — similar pair   : {sim_similar:.4f}')
print(f'Cosine(s1, s3) — dissimilar pair: {sim_dissimilar:.4f}')
print(f'Embedding dimension: {len(e1)}')

## 3. InMemoryVectorIndex — add 5 chunks, search for "revenue", show top-3

In [ ]:
embedder = rag.RandomEmbedder(dim=64)

docs = [
    'Revenue reached $12.4 billion, a record high for the company.',
    'Operating margins improved to 28% year-over-year.',
    'Revenue growth was driven by the cloud division expanding globally.',
    'Headcount increased by 3% as part of the expansion strategy.',
    'Q3 revenue beat analyst expectations by a wide margin.',
]

chunks = []
for i, text in enumerate(docs):
    c = rag.Chunk(text=text, source='report', chunk_id=f'doc::{i}', chunk_index=i)
    c.embedding = embedder.embed(text)
    chunks.append(c)

index = rag.InMemoryVectorIndex()
index.add_batch(chunks)

query_emb = embedder.embed('revenue')
results = index.search(query_emb, k=3)

print('Top-3 results for query "revenue":')
for rank, (chunk, score) in enumerate(results, 1):
    print(f'  {rank}. score={score:.4f} | {chunk.text[:70]}')

## 4. BM25Index — add same 5 chunks, search "revenue growth", show BM25 scores

In [ ]:
# Re-use the same 'chunks' list from section 3 (no embeddings required for BM25)
bm25 = rag.BM25Index(k1=1.5, b=0.75)
bm25.add_batch(chunks)

bm25_results = bm25.search('revenue growth', k=5)

print('BM25 results for "revenue growth" (all 5 docs):')
for rank, (chunk, score) in enumerate(bm25_results, 1):
    print(f'  {rank}. score={score:.4f} | {chunk.text[:70]}')

## 5. HybridIndex — compare hybrid vs pure-vector results

In [ ]:
hybrid = rag.HybridIndex(vector_weight=0.6, bm25_weight=0.4)
hybrid.add_batch(chunks)

query_text = 'revenue growth'
query_emb  = embedder.embed(query_text)

hybrid_results = hybrid.search(query_emb, query_text, k=3)
vector_results = index.search(query_emb, k=3)

print('Hybrid top-3  (vector 60% + BM25 40%):')
for rank, (c, s) in enumerate(hybrid_results, 1):
    print(f'  {rank}. score={s:.4f} | {c.text[:65]}')

print('\nPure-vector top-3:')
for rank, (c, s) in enumerate(vector_results, 1):
    print(f'  {rank}. score={s:.4f} | {c.text[:65]}')

## 6. CitationTracker + RetrievalFeedback — record, mark useless, verify weight decay

In [ ]:
tracker  = rag.CitationTracker()
feedback = rag.RetrievalFeedback(boost=1.1, decay=0.9)

# Record citations for run-1
c0, c1, c2 = chunks[0], chunks[1], chunks[2]

cite0 = tracker.record(c0, score=0.91, run_id='run-1')
cite1 = tracker.record(c1, score=0.74, run_id='run-1')
cite2 = tracker.record(c2, score=0.55, run_id='run-1')

print('Citations for run-1:')
for cite in tracker.for_run('run-1'):
    print(f'  chunk_id={cite.chunk_id}  score={cite.relevance_score:.2f}  excerpt={cite.excerpt[:50]}')

print(f'\nchunk[1] weight before feedback: {c1.weight:.4f}')
feedback.mark_useless(c1)
print(f'chunk[1] weight after mark_useless: {c1.weight:.4f}  (decay={feedback.decay})')

print(f'\nchunk[0] weight before feedback: {c0.weight:.4f}')
feedback.mark_useful(c0)
print(f'chunk[0] weight after mark_useful : {c0.weight:.4f}  (boost={feedback.boost})')

## 7. RAGPipeline end-to-end — ingest, retrieve_and_cite, print context + citations

In [ ]:
ANNUAL_REPORT = """
FY2024 Annual Report — TechCorp Inc.

Revenue for fiscal year 2024 totalled $48.7 billion, an increase of 14% from FY2023.
Cloud services remained the fastest-growing segment, contributing $18.2 billion.

Operating income rose to $11.3 billion, reflecting improved efficiency across divisions.
The company hired 4,200 engineers and expanded into three new markets.

Capital allocation: $2.1 billion in R&D, $800 million in capex, and a $1 billion buyback.
The board declared a quarterly dividend of $0.45 per share.
"""

pipeline = rag.RAGPipeline(
    chunker=rag.FixedChunker(chunk_size=200, overlap=30),
    embedder=rag.RandomEmbedder(dim=64),
    hybrid=True,
    feedback=True,
)

ingested = pipeline.ingest(ANNUAL_REPORT, source='annual_report_2024')
print(f'Ingested {pipeline.n_chunks} chunks from annual report')

context, citations = pipeline.retrieve_and_cite(
    query='What is revenue?',
    n=3,
    run_id='qa-run-1',
)

print('\n=== Retrieved context ===')
print(context)

print('\n=== Citations ===')
for i, c in enumerate(citations, 1):
    print(f'  [{i}] source={c.source}  chunk_id={c.chunk_id}  score={c.relevance_score:.4f}')